In [3]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_customers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_sellers_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_reviews_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_items_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_products_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_geolocation_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/product_category_name_translation.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_orders_dataset.csv
/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce/olist_order_payments_dataset.csv


In [4]:
# --- BƯỚC 1: CÀI ĐẶT THƯ VIỆN & CẤU HÌNH ---
import pandas as pd
import numpy as np
import os

print("✅ Đã import Pandas và NumPy!")

# --- BƯỚC 2: LOAD DỮ LIỆU TRÊN KAGGLE ---
# Lưu ý: Đảm bảo bạn đã click "Add Data" và add dataset "Brazilian E-Commerce Public Dataset by Olist" vào notebook.
# Đường dẫn mặc định trên Kaggle thường là:
data_folder = "/kaggle/input/datasets/organizations/olistbr/brazilian-ecommerce" 

FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv"
}

print("\n📥 Loading dữ liệu từ Kaggle Input...")
dfs = {}
for key, filename in FILES.items():
    filepath = os.path.join(data_folder, filename)
    if os.path.exists(filepath):
        dfs[key] = pd.read_csv(filepath)
        print(f"  ✅ Loaded {key}: {len(dfs[key])} rows")
    else:
        print(f"  ❌ Missing {filename} - Vui lòng kiểm tra lại đường dẫn Add Data")



✅ Đã import Pandas và NumPy!

📥 Loading dữ liệu từ Kaggle Input...
  ✅ Loaded orders: 99441 rows
  ✅ Loaded customers: 99441 rows
  ✅ Loaded items: 112650 rows
  ✅ Loaded payments: 103886 rows
  ✅ Loaded reviews: 99224 rows
  ✅ Loaded products: 32951 rows
  ✅ Loaded sellers: 3095 rows
  ✅ Loaded geolocation: 1000163 rows
  ✅ Loaded translation: 71 rows


In [5]:
# --- BƯỚC 3: TIỀN XỬ LÝ & JOIN DỮ LIỆU ---
print("\n🔧 Bắt đầu tiền xử lý...")

# 1. Xử lý geolocation: lấy tọa độ trung bình theo zip_code
df_geo_clean = dfs["geolocation"].groupby("geolocation_zip_code_prefix", as_index=False).agg(
    lat_avg=("geolocation_lat", "mean"),
    lng_avg=("geolocation_lng", "mean")
).round(4)

# 2. Join các bảng (Master Join)
df_master = dfs["orders"]
df_master = df_master.merge(dfs["customers"], on="customer_id", how="left")
df_master = df_master.merge(
    df_geo_clean, 
    left_on="customer_zip_code_prefix", 
    right_on="geolocation_zip_code_prefix", 
    how="left"
).drop(columns=["geolocation_zip_code_prefix"])

df_master = df_master.merge(dfs["items"], on="order_id", how="left")
df_master = df_master.merge(dfs["products"], on="product_id", how="left")
df_master = df_master.merge(dfs["translation"], on="product_category_name", how="left")
df_master = df_master.merge(dfs["sellers"], on="seller_id", how="left")
df_master = df_master.merge(dfs["payments"], on="order_id", how="left")
df_master = df_master.merge(dfs["reviews"], on="order_id", how="left")

print(f"  ✅ Đã join 9 bảng. Tổng bản ghi: {len(df_master)}")


🔧 Bắt đầu tiền xử lý...
  ✅ Đã join 9 bảng. Tổng bản ghi: 119143


In [6]:
# --- BƯỚC 4: XỬ LÝ MISSING VALUES & FEATURE ENGINEERING ---
df_master["review_comment_message"] = df_master["review_comment_message"].fillna("No Comment")
df_master["review_score"] = df_master["review_score"].fillna(0)

timestamp_cols = [
    "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date", "review_creation_date",
    "review_answer_timestamp", "shipping_limit_date"
]

for col in timestamp_cols:
    if col in df_master.columns:
        df_master[col] = pd.to_datetime(df_master[col], errors="coerce")

if "order_delivered_customer_date" in df_master.columns and "order_estimated_delivery_date" in df_master.columns:
    df_master["delivery_delay_days"] = (df_master["order_delivered_customer_date"] - df_master["order_estimated_delivery_date"]).dt.days

In [7]:
# --- BƯỚC 5: KIỂM TRA KẾT QUẢ ---
print("\n📋 Kết quả sau tiền xử lý:")
print(f"  • Tổng columns: {len(df_master.columns)}")
print(f"  • Tổng rows: {len(df_master)}")


# --- BƯỚC 6: LƯU DỮ LIỆU ĐÃ XỬ LÝ VÀO KAGGLE WORKING ---
print("\n💾 Đang chuẩn bị lưu dữ liệu...")

# Lưu vào /kaggle/working/ để file được giữ lại sau khi chạy xong (Output)
SAVE_PATH = "/kaggle/working/processed_data/week1"
os.makedirs(SAVE_PATH, exist_ok=True)

df_master.to_parquet(f"{SAVE_PATH}/master_dataset.parquet", index=False)
print(f"  ✅ Đã lưu dataset vào: {SAVE_PATH}/master_dataset.parquet")

with open(f"{SAVE_PATH}/schema_info.txt", "w") as f:
    df_master.info(buf=f)
print(f"  ✅ Đã lưu schema vào: {SAVE_PATH}/schema_info.txt")


📋 Kết quả sau tiền xử lý:
  • Tổng columns: 43
  • Tổng rows: 119143

💾 Đang chuẩn bị lưu dữ liệu...
  ✅ Đã lưu dataset vào: /kaggle/working/processed_data/week1/master_dataset.parquet
  ✅ Đã lưu schema vào: /kaggle/working/processed_data/week1/schema_info.txt
